In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "290_build_silver_fact_estimate_project.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.silver.fact_estimate_item"
DIM_PROJECT_TABLE = f"{catalog}.silver.dim_project"
DIM_PROJECT_CLASSIFICATION_TABLE = f"{catalog}.silver.dim_project_classification"
DIM_COUNTY_TABLE = f"{catalog}.silver.dim_county"
TARGET_TABLE = f"{catalog}.silver.fact_estimate_project"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build fact_estimate_project
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    USING DELTA
    AS
    WITH project_rollup AS (
        SELECT
              project_id
            , MIN(project_key) AS project_key

            , COUNT(*) AS estimate_item_count
            , COUNT(DISTINCT item_specification_key) AS distinct_item_specification_count

            , SUM(COALESCE(extended_estimate_item_amount_calc, 0)) AS estimate_item_extended_total

            , MIN(engineer_project_total) AS min_engineer_project_total
            , MAX(engineer_project_total) AS max_engineer_project_total

            , MAX(CASE WHEN estimate_item_changed_across_versions THEN 1 ELSE 0 END) = 1 AS has_item_version_changes

            , SUM(CASE WHEN is_payment_adjustment_item THEN 1 ELSE 0 END) AS payment_adjustment_item_count
            , SUM(CASE WHEN is_special_deduction_item THEN 1 ELSE 0 END) AS special_deduction_item_count

            , MAX(source_updated_at) AS latest_source_updated_at
            , MAX(ingested_at_utc) AS latest_ingested_at_utc

        FROM {SOURCE_TABLE}
        GROUP BY project_id
    )

    SELECT
          md5(COALESCE(p.project_id, '')) AS estimate_project_fact_key

        , p.project_key
        , p.project_id

        , pc.project_classification_key
        , dc.county_key

        , dp.project_name
        , dp.project_number
        , dp.state_project_number
        , dp.control_section_job_csj
        , dp.controlling_project_id_ccsj
        , dp.project_type
        , dp.project_classification
        , dp.project_actual_let_date
        , dp.project_estimated_let_date
        , dp.project_limits_from
        , dp.project_limits_to
        , dp.county
        , dc.county_location AS location
        , dp.district_division
        , dp.highway
        , dp.short_description
        , dp.federal_project_number

        , p.estimate_item_count
        , p.distinct_item_specification_count
        , p.estimate_item_extended_total

        , p.min_engineer_project_total
        , p.max_engineer_project_total

        , CASE
              WHEN p.min_engineer_project_total IS NOT NULL
               AND p.max_engineer_project_total IS NOT NULL
               AND p.min_engineer_project_total <> p.max_engineer_project_total
              THEN TRUE
              ELSE FALSE
          END AS has_conflicting_engineer_project_totals

        , CASE
              WHEN p.max_engineer_project_total IS NOT NULL
               AND p.estimate_item_extended_total IS NOT NULL
              THEN ABS(p.max_engineer_project_total - p.estimate_item_extended_total)
              ELSE NULL
          END AS estimate_total_vs_item_rollup_abs_diff

        , p.has_item_version_changes
        , p.payment_adjustment_item_count
        , p.special_deduction_item_count

        , p.latest_source_updated_at
        , p.latest_ingested_at_utc

    FROM project_rollup p
    LEFT JOIN {DIM_PROJECT_TABLE} dp
        ON p.project_id = dp.project_id
    LEFT JOIN {DIM_PROJECT_CLASSIFICATION_TABLE} pc
        ON (
            CASE
                WHEN COALESCE(dp.project_classification, '') = '' THEN 'UNKNOWN'
                ELSE dp.project_classification
            END
        ) = pc.project_classification
    LEFT JOIN {DIM_COUNTY_TABLE} dc
        ON (
            CASE
                WHEN COALESCE(dp.county, '') = '' THEN 'UNKNOWN'
                WHEN dp.county = 'De Witt' THEN 'DeWitt'
                ELSE dp.county
            END
        ) = dc.county
    """)

    spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE} IS
    'Project-level engineer estimate fact table built from silver.fact_estimate_item and linked to project, classification, and county dimensions.'
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_TABLE}").collect()[0]["row_count"]

    log_run(
        "SUCCESS",
        row_count,
        f"Built {TARGET_TABLE} successfully."
    )

    print("=" * 70)
    print(f"Built {TARGET_TABLE}")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise